# Diffusion Grid Search Tuning, Debug Version

This rewritten notebook adds the debugging/sanity changes discussed:

- Train at `64x64` first before scaling to `256x256`
- Optional tiny-overfit mode using 32 images
- Save 4 generated images every 50 epochs
- Save 4 real images next to generated images
- Print sampling statistics before clamping so white/saturated samples are easier to diagnose
- Use a reduced config list while debugging


In [42]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("DEVICE:", DEVICE)
print("CUDA available:", torch.cuda.is_available())

x = torch.randn(2, 3).to(DEVICE)
print(x)
print("CUDA test passed")

DEVICE: cuda
CUDA available: True
tensor([[ 1.2463,  2.4726, -0.5153],
        [ 1.1351, -0.1045,  1.2728]], device='cuda:0')
CUDA test passed


In [70]:
# -------------------------
# Imports and global settings
# -------------------------

import os
import json
import math
import time
import random
import itertools

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.utils import save_image
from datasets import load_from_disk
from tqdm.auto import tqdm

# -------------------------
# Main settings
# -------------------------

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

DATA_DIR = "processed_fish_256"

# Start smaller first. Once this works, change IMG_SIZE back to 256.
IMG_SIZE = 64
TIMESTEPS = 500
EPOCHS = 1000

# -------------------------
# Debug / sanity settings
# -------------------------

DEBUG_SAMPLE_STATS = False
OVERFIT_TINY_SUBSET = False
TINY_SUBSET_SIZE = 32
SUBSET_FRACTION = 1
SAVE_EVERY = 50
NUM_SAMPLES_TO_SAVE = 4

RUN_FULL_GRID = True

if RUN_FULL_GRID:
    param_grid = {
        "lr": [5e-5, 1e-4],
        "batch_size": [16],
        "weight_decay": [1e-4],
        "grad_clip": [1.0],
    }
else:
    param_grid = {
        "lr": [1e-4],
        "batch_size": [16],
        "weight_decay": [1e-4],
        "grad_clip": [1.0],
    }

keys = list(param_grid.keys())
configs = [dict(zip(keys, values)) for values in itertools.product(*param_grid.values())]

print(f"Total configs: {len(configs)}")
print(configs)


Using device: cuda
GPU: NVIDIA H100 80GB HBM3
Total configs: 2
[{'lr': 5e-05, 'batch_size': 16, 'weight_decay': 0.0001, 'grad_clip': 1.0}, {'lr': 0.0001, 'batch_size': 16, 'weight_decay': 0.0001, 'grad_clip': 1.0}]


## Load and prepare the dataset

This cell keeps the cached image tensors on CPU and moves each batch to the GPU inside the training loop. That is usually safer than storing the whole dataset on the GPU, especially when changing image size or dataset size.


In [71]:
# -------------------------
# Dataset loading
# -------------------------

TENSOR_CACHE = f"fish_{IMG_SIZE}_normalized.pt"

if os.path.exists(TENSOR_CACHE):
    print(f"Loading cached tensor from {TENSOR_CACHE}...")
    all_images = torch.load(TENSOR_CACHE)
    print(f"Loaded: {tuple(all_images.shape)}")
else:
    print("Loading from HuggingFace format (one-time)...")
    raw = load_from_disk(DATA_DIR)
    raw.set_format(None)
    print(f"Full dataset size: {len(raw)}")

    random.seed(0)

    if OVERFIT_TINY_SUBSET:
        subset_size = min(TINY_SUBSET_SIZE, len(raw))
        subset_indices = list(range(subset_size))
        print(f"Using tiny overfit dataset with {subset_size} images")
    else:
        subset_size = max(1, int(len(raw) * SUBSET_FRACTION))
        subset_indices = random.sample(range(len(raw)), subset_size)
        print(f"Using {SUBSET_FRACTION:.0%} subset with {subset_size} images")

    chunks = []
    chunk_size = 500
    for start in tqdm(range(0, subset_size, chunk_size), desc="Loading subset"):
        end = min(start + chunk_size, subset_size)
        batch_indices = subset_indices[start:end]
        batch = raw.select(batch_indices)["pixel_values"]
        chunks.append(np.array(batch))

    all_images = np.concatenate(chunks, axis=0)
    all_images = torch.tensor(all_images, dtype=torch.float32)

    if all_images.shape[-1] == 3:
        all_images = all_images.permute(0, 3, 1, 2)

    if all_images.max() > 2.0:
        all_images = all_images / 127.5 - 1.0
    else:
        all_images = all_images * 2.0 - 1.0

    if all_images.shape[-1] != IMG_SIZE or all_images.shape[-2] != IMG_SIZE:
        all_images = F.interpolate(
            all_images, size=(IMG_SIZE, IMG_SIZE),
            mode="bilinear", align_corners=False,
        )

    all_images = all_images.contiguous()
    torch.save(all_images, TENSOR_CACHE)
    print(f"Saved cache to {TENSOR_CACHE}")

# -------------------------
# Move to GPU if it fits
# -------------------------

gb = all_images.numel() * all_images.element_size() / 1e9
print(f"Dataset size: {gb:.2f} GB")

if torch.cuda.is_available() and gb < 4.0:
    all_images = all_images.to(DEVICE)
    print(f"Dataset on GPU — zero transfer overhead per batch")
else:
    print("Dataset stays on CPU — will transfer per batch")

print(f"Shape: {tuple(all_images.shape)}")
print(f"Min/max: {all_images.min().item():.3f} / {all_images.max().item():.3f}")
print(f"Mean/std: {all_images.mean().item():.3f} / {all_images.std().item():.3f}")

# -------------------------
# Dataset wrapper
# -------------------------

class CachedDataset(Dataset):
    def __init__(self, images):
        self.images = images

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx]

grid_dataset = CachedDataset(all_images)
print(f"Grid dataset size: {len(grid_dataset)}")

Loading cached tensor from fish_64_normalized.pt...
Loaded: (3990, 3, 64, 64)
Dataset size: 0.20 GB
Dataset on GPU — zero transfer overhead per batch
Shape: (3990, 3, 64, 64)
Min/max: -1.000 / 1.000
Mean/std: -0.056 / 0.568
Grid dataset size: 3990


In [72]:
# -------------------------
# Dataset sanity check
# -------------------------

example = grid_dataset[0]
print("Example type:", type(example))
print("Example dtype:", example.dtype)
print("Example shape:", tuple(example.shape))
print("Example min/max:", example.min().item(), example.max().item())
print("Example mean/std:", example.mean().item(), example.std().item())

# Save a few real examples before training.
os.makedirs("debug_samples", exist_ok=True)
real_preview = (all_images[:4].clamp(-1, 1) + 1) / 2
save_image(real_preview, "debug_samples/real_preview_before_training.png", nrow=2)
print("Saved debug_samples/real_preview_before_training.png")


Example type: <class 'torch.Tensor'>
Example dtype: torch.float32
Example shape: (3, 64, 64)
Example min/max: -1.0 0.656862735748291
Example mean/std: 0.07634583115577698 0.5546301007270813
Saved debug_samples/real_preview_before_training.png


## Diffusion schedule, noise function, and U-Net

In [73]:
# -------------------------
# Offset cosine diffusion schedule
# -------------------------

def offset_cosine_beta_schedule(timesteps, s=0.008):
    steps = timesteps + 1
    x = torch.linspace(0, timesteps, steps)

    alpha_hats = torch.cos(
        ((x / timesteps) + s) / (1 + s) * math.pi * 0.5
    ) ** 2

    alpha_hats = alpha_hats / alpha_hats[0]
    betas = 1 - (alpha_hats[1:] / alpha_hats[:-1])
    betas = torch.clip(betas, 1e-4, 0.999)
    return betas

betas = offset_cosine_beta_schedule(TIMESTEPS).to(DEVICE)
alphas = (1.0 - betas).to(DEVICE)
alpha_hats = torch.cumprod(alphas, dim=0).to(DEVICE)


def add_noise(x, t):
    noise = torch.randn_like(x)
    alpha_hat = alpha_hats[t].view(-1, 1, 1, 1)
    noisy_x = torch.sqrt(alpha_hat) * x + torch.sqrt(1 - alpha_hat) * noise
    return noisy_x, noise


# -------------------------
# Time embedding
# -------------------------

class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half_dim = self.dim // 2
        emb_scale = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=t.device) * -emb_scale)
        emb = t[:, None].float() * emb[None, :]
        return torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)


# -------------------------
# U-Net blocks
# -------------------------

class SelfAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.norm = nn.GroupNorm(8, channels)
        self.qkv = nn.Conv2d(channels, channels * 3, 1)
        self.proj = nn.Conv2d(channels, channels, 1)
        self.scale = channels ** -0.5

    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x)
        qkv = self.qkv(h).reshape(B, 3, C, H * W)
        q, k, v = qkv[:, 0], qkv[:, 1], qkv[:, 2]
        attn = torch.softmax(q.transpose(-1, -2) @ k * self.scale, dim=-1)
        out = (v @ attn.transpose(-1, -2)).reshape(B, C, H, W)
        return x + self.proj(out)


class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.time_mlp = nn.Linear(time_dim, out_channels)

        self.norm1 = nn.GroupNorm(8, out_channels)
        self.norm2 = nn.GroupNorm(8, out_channels)

        if in_channels != out_channels:
            self.skip = nn.Conv2d(in_channels, out_channels, 1)
        else:
            self.skip = nn.Identity()

    def forward(self, x, t):
        h = self.conv1(x)
        h = self.norm1(h)
        h = F.silu(h)

        time_emb = self.time_mlp(t)
        h = h + time_emb[:, :, None, None]

        h = self.conv2(h)
        h = self.norm2(h)
        h = F.silu(h)

        return h + self.skip(x)


class UNet(nn.Module):
    def __init__(self, img_channels=3, base_channels=64, time_dim=256):
        super().__init__()

        self.time_embedding = nn.Sequential(
            SinusoidalTimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim),
        )

        c1 = base_channels
        c2 = base_channels * 2
        c3 = base_channels * 4
        c4 = base_channels * 8

        self.enc1 = ResBlock(img_channels, c1, time_dim)
        self.enc2 = ResBlock(c1, c2, time_dim)
        self.enc3 = ResBlock(c2, c3, time_dim)
        self.enc4 = ResBlock(c3, c4, time_dim)

        self.down1 = nn.Conv2d(c1, c1, 3, stride=2, padding=1)
        self.down2 = nn.Conv2d(c2, c2, 3, stride=2, padding=1)
        self.down3 = nn.Conv2d(c3, c3, 3, stride=2, padding=1)
        self.down4 = nn.Conv2d(c4, c4, 3, stride=2, padding=1)

        # Attention on the low-res feature maps
        self.attn_enc3 = SelfAttention(c3)
        self.attn_enc4 = SelfAttention(c4)
        self.attn_bot  = SelfAttention(c4)
        self.attn_dec4 = SelfAttention(c3)
        self.attn_dec3 = SelfAttention(c2)

        self.bottleneck = ResBlock(c4, c4, time_dim)

        self.up4 = nn.ConvTranspose2d(c4, c4, kernel_size=2, stride=2)
        self.dec4 = ResBlock(c4 + c4, c3, time_dim)

        self.up3 = nn.ConvTranspose2d(c3, c3, kernel_size=2, stride=2)
        self.dec3 = ResBlock(c3 + c3, c2, time_dim)

        self.up2 = nn.ConvTranspose2d(c2, c2, kernel_size=2, stride=2)
        self.dec2 = ResBlock(c2 + c2, c1, time_dim)

        self.up1 = nn.ConvTranspose2d(c1, c1, kernel_size=2, stride=2)
        self.dec1 = ResBlock(c1 + c1, c1, time_dim)

        self.out = nn.Conv2d(c1, img_channels, kernel_size=1)

    def forward(self, x, t):
        t = self.time_embedding(t)

        e1 = self.enc1(x, t)
        e2 = self.enc2(self.down1(e1), t)
        e3 = self.attn_enc3(self.enc3(self.down2(e2), t))
        e4 = self.attn_enc4(self.enc4(self.down3(e3), t))

        b = self.attn_bot(self.bottleneck(self.down4(e4), t))

        d4 = self.up4(b)
        d4 = torch.cat([d4, e4], dim=1)
        d4 = self.attn_dec4(self.dec4(d4, t))

        d3 = self.up3(d4)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.attn_dec3(self.dec3(d3, t))

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2, t)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1, t)

        return self.out(d1)

## Sampling and saving helpers

The sampling function prints the range of the image before clamping. If values are far outside `[-1, 1]`, the reverse process is unstable and the final saved PNG may look white because of clamping.


In [74]:
# -------------------------
# Sampling
# -------------------------

@torch.no_grad()
def sample(model, n=4, debug=False):
    model.eval()
    x = torch.randn(n, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)

    for i in reversed(range(TIMESTEPS)):
        t = torch.full((n,), i, device=DEVICE, dtype=torch.long)
        predicted_noise = model(x, t)

        alpha = alphas[t].view(-1, 1, 1, 1)
        alpha_hat = alpha_hats[t].view(-1, 1, 1, 1)
        beta = betas[t].view(-1, 1, 1, 1)

        if i > 0:
            noise = torch.randn_like(x)
        else:
            noise = torch.zeros_like(x)

        x = (1 / torch.sqrt(alpha)) * (
            x - ((1 - alpha) / torch.sqrt(1 - alpha_hat)) * predicted_noise
        ) + torch.sqrt(beta) * noise

        x = torch.clamp(x, -1.5, 1.5)

        if debug and i % 100 == 0:
            print(
                f"t={i:04d} | "
                f"x min={x.min().item():.3f} | "
                f"x max={x.max().item():.3f} | "
                f"x mean={x.mean().item():.3f} | "
                f"x std={x.std().item():.3f} | "
                f"pred_noise std={predicted_noise.std().item():.3f}"
            )

    if debug:
        print(
            "Before clamp:",
            "min=", x.min().item(),
            "max=", x.max().item(),
            "mean=", x.mean().item(),
            "std=", x.std().item(),
        )

    # Convert expected image range [-1, 1] to [0, 1] for saving.
    x = torch.clamp(x, -1, 1)
    x = (x + 1) / 2
    return x


def save_samples(model, run_id, epoch, config, n=4, save_dir="grid_samples"):
    os.makedirs(save_dir, exist_ok=True)

    samples = sample(model, n=n, debug=DEBUG_SAMPLE_STATS)

    filename = (
        f"generated_"
        f"run_{run_id:02d}_"
        f"epoch_{epoch:03d}_"
        f"lr_{config['lr']}_"
        f"bs_{config['batch_size']}_"
        f"wd_{config['weight_decay']}_"
        f"gc_{config['grad_clip']}.png"
    )

    save_path = os.path.join(save_dir, filename)
    save_image(samples, save_path, nrow=2)
    print(f"Saved generated samples to {save_path}")


def save_real_samples(loader, epoch, save_dir, n=4):
    os.makedirs(save_dir, exist_ok=True)

    batch = next(iter(loader))
    if isinstance(batch, (list, tuple)):
        images = batch[0]
    else:
        images = batch

    images = images[:n]
    images_for_save = (images.cpu().clamp(-1, 1) + 1) / 2

    save_path = os.path.join(save_dir, f"real_epoch_{epoch:03d}.png")
    save_image(images_for_save, save_path, nrow=2)
    print(f"Saved real samples to {save_path}")
    print(
        "Real batch stats:",
        "min=", images.min().item(),
        "max=", images.max().item(),
        "mean=", images.mean().item(),
        "std=", images.std().item(),
    )


## Training loop

In [75]:
# -------------------------
# Training function
# -------------------------

def train_one_config(config, run_id):
    run_name = (
        f"run_{run_id:03d}_"
        f"lr{config['lr']}_"
        f"bs{config['batch_size']}_"
        f"wd{config['weight_decay']}_"
        f"clip{config['grad_clip']}"
    )

    sample_dir = os.path.join("gridsearch_samples", run_name)
    os.makedirs(sample_dir, exist_ok=True)

    model = UNet().to(DEVICE)

    # ---- EMA piece 1: initialize after creating model ----
    ema_weights = {k: v.clone() for k, v in model.state_dict().items()}
    ema_decay = 0.9999

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )

    scaler = torch.amp.GradScaler("cuda", enabled=False)

    loader = DataLoader(
        grid_dataset,
        batch_size=config["batch_size"],
        shuffle=True,
        num_workers=0,
        pin_memory=False,
    )

    save_real_samples(loader, epoch=0, save_dir=sample_dir, n=NUM_SAMPLES_TO_SAVE)

    epoch_losses = []
    best_epoch_loss = float("inf")
    start_run = time.time()

    total_steps = EPOCHS * len(loader)
    pbar = tqdm(total=total_steps, desc=f"Run {run_id}/{len(configs)}", leave=True)

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0.0

        for images in loader:
            if isinstance(images, (list, tuple)):
                images = images[0]

            images = images.to(DEVICE, non_blocking=True)

            t = torch.randint(
                0, TIMESTEPS,
                (images.size(0),),
                device=DEVICE, dtype=torch.long,
            )

            with torch.amp.autocast("cuda", enabled=False):
                noisy_images, noise = add_noise(images, t)
                predicted_noise = model(noisy_images, t)
                loss = F.mse_loss(predicted_noise, noise)

            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(), max_norm=config["grad_clip"],
            )

            scaler.step(optimizer)
            scaler.update()

            # ---- EMA piece 2: update after each training step ----
            with torch.no_grad():
                for k, v in model.state_dict().items():
                    ema_weights[k].lerp_(v, 1 - ema_decay)

            total_loss += loss.item()
            pbar.set_postfix({"epoch": epoch + 1, "loss": f"{loss.item():.4f}"})
            pbar.update(1)

        avg_loss = total_loss / len(loader)
        epoch_losses.append(avg_loss)

        if avg_loss < best_epoch_loss:
            best_epoch_loss = avg_loss
            torch.save(model.state_dict(), os.path.join(sample_dir, "best_model.pt"))

        if (epoch + 1) % SAVE_EVERY == 0 or (epoch + 1) == EPOCHS:

            # ---- EMA piece 3: swap weights before sampling, restore after ----
            original = {k: v.clone() for k, v in model.state_dict().items()}
            model.load_state_dict(ema_weights)

            save_samples(
                model=model, run_id=run_id, epoch=epoch + 1,
                config=config, n=NUM_SAMPLES_TO_SAVE, save_dir=sample_dir,
            )

            model.load_state_dict(original)
            print(f"\nEpoch {epoch + 1}: avg_loss={avg_loss:.6f}")

            save_real_samples(
                loader=loader,
                epoch=epoch + 1,
                save_dir=sample_dir,
                n=NUM_SAMPLES_TO_SAVE,
            )

    pbar.close()
    total_time = time.time() - start_run

    result = {
        "run_id": run_id,
        "lr": config["lr"],
        "batch_size": config["batch_size"],
        "weight_decay": config["weight_decay"],
        "grad_clip": config["grad_clip"],
        "final_loss": epoch_losses[-1],
        "best_epoch_loss": min(epoch_losses),
        "last_3_avg_loss": sum(epoch_losses[-3:]) / min(3, len(epoch_losses)),
        "epoch_losses": epoch_losses,
        "runtime_minutes": total_time / 60,
        "sample_dir": sample_dir,
    }

    # Also save final weights for inspection.
    torch.save(model.state_dict(), os.path.join(sample_dir, "final_model.pt"))

    del model
    del optimizer
    del scaler
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    return result


## Quick sanity check before running the full training loop

This checks that real images, noisy images, and predicted noise have reasonable ranges. The model is untrained here, so the predicted noise will not be good yet, but the shapes/ranges should make sense.


In [76]:
# -------------------------
# Quick sanity check with an untrained model
# -------------------------

sanity_loader = DataLoader(
    grid_dataset,
    batch_size=min(4, len(grid_dataset)),
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

sanity_model = UNet().to(DEVICE)

with torch.no_grad():
    real = next(iter(sanity_loader)).to(DEVICE)
    t = torch.randint(
    low=0,
    high=TIMESTEPS,
    size=(real.size(0),),
    device=DEVICE,
    dtype=torch.long
    )
    noisy, actual_noise = add_noise(real, t)
    pred_noise = sanity_model(noisy, t)

    print("real shape:       ", tuple(real.shape))
    print("real range:       ", real.min().item(), real.max().item())
    print("noisy range:      ", noisy.min().item(), noisy.max().item())
    print("actual noise std: ", actual_noise.std().item())
    print("pred noise range: ", pred_noise.min().item(), pred_noise.max().item())
    print("pred noise std:   ", pred_noise.std().item())
    print("betas device:     ", betas.device)
    print("alpha_hats device:", alpha_hats.device)

del sanity_model
if DEVICE == "cuda":
    torch.cuda.empty_cache()


real shape:        (4, 3, 64, 64)
real range:        -1.0 1.0
noisy range:       -3.6252994537353516 3.9154324531555176
actual noise std:  0.9974072575569153
pred noise range:  -1.4333170652389526 2.144148826599121
pred noise std:    0.3495792746543884
betas device:      cuda:0
alpha_hats device: cuda:0


## Run training / grid search

In [77]:
# -------------------------
# Run training / grid search
# -------------------------

all_results = []

for run_id, config in enumerate(configs, start=1):
    result = train_one_config(config, run_id)
    all_results.append(result)

    # Save after every run in case the notebook stops.
    with open("gridsearch_loss_results.json", "w") as f:
        json.dump(all_results, f, indent=2)

    df = pd.DataFrame(all_results)
    df.to_csv("gridsearch_loss_results.csv", index=False)

print("Training complete.")


Saved real samples to gridsearch_samples/run_001_lr5e-05_bs16_wd0.0001_clip1.0/real_epoch_000.png
Real batch stats: min= -1.0 max= 1.0 mean= -0.10972252488136292 std= 0.4846981167793274


Run 1/2:   0%|          | 0/250000 [00:00<?, ?it/s]

Saved generated samples to gridsearch_samples/run_001_lr5e-05_bs16_wd0.0001_clip1.0/generated_run_01_epoch_050_lr_5e-05_bs_16_wd_0.0001_gc_1.0.png

Epoch 50: avg_loss=0.054307
Saved real samples to gridsearch_samples/run_001_lr5e-05_bs16_wd0.0001_clip1.0/real_epoch_050.png
Real batch stats: min= -1.0 max= 1.0 mean= -0.18620797991752625 std= 0.4955938458442688
Saved generated samples to gridsearch_samples/run_001_lr5e-05_bs16_wd0.0001_clip1.0/generated_run_01_epoch_100_lr_5e-05_bs_16_wd_0.0001_gc_1.0.png

Epoch 100: avg_loss=0.052472
Saved real samples to gridsearch_samples/run_001_lr5e-05_bs16_wd0.0001_clip1.0/real_epoch_100.png
Real batch stats: min= -1.0 max= 1.0 mean= -0.18490658700466156 std= 0.42543232440948486
Saved generated samples to gridsearch_samples/run_001_lr5e-05_bs16_wd0.0001_clip1.0/generated_run_01_epoch_150_lr_5e-05_bs_16_wd_0.0001_gc_1.0.png

Epoch 150: avg_loss=0.044137
Saved real samples to gridsearch_samples/run_001_lr5e-05_bs16_wd0.0001_clip1.0/real_epoch_150.png

Run 2/2:   0%|          | 0/250000 [00:00<?, ?it/s]

Saved generated samples to gridsearch_samples/run_002_lr0.0001_bs16_wd0.0001_clip1.0/generated_run_02_epoch_050_lr_0.0001_bs_16_wd_0.0001_gc_1.0.png

Epoch 50: avg_loss=0.052290
Saved real samples to gridsearch_samples/run_002_lr0.0001_bs16_wd0.0001_clip1.0/real_epoch_050.png
Real batch stats: min= -0.9960784316062927 max= 1.0 mean= -0.04341334104537964 std= 0.6299009323120117
Saved generated samples to gridsearch_samples/run_002_lr0.0001_bs16_wd0.0001_clip1.0/generated_run_02_epoch_100_lr_0.0001_bs_16_wd_0.0001_gc_1.0.png

Epoch 100: avg_loss=0.048635
Saved real samples to gridsearch_samples/run_002_lr0.0001_bs16_wd0.0001_clip1.0/real_epoch_100.png
Real batch stats: min= -1.0 max= 1.0 mean= -0.21626530587673187 std= 0.5095347166061401
Saved generated samples to gridsearch_samples/run_002_lr0.0001_bs16_wd0.0001_clip1.0/generated_run_02_epoch_150_lr_0.0001_bs_16_wd_0.0001_gc_1.0.png

Epoch 150: avg_loss=0.047073
Saved real samples to gridsearch_samples/run_002_lr0.0001_bs16_wd0.0001_cli

In [78]:
# -------------------------
# Results table
# -------------------------

results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values("last_3_avg_loss")

results_df[
    [
        "run_id",
        "lr",
        "batch_size",
        "weight_decay",
        "grad_clip",
        "final_loss",
        "best_epoch_loss",
        "last_3_avg_loss",
        "runtime_minutes",
        "sample_dir",
    ]
]


,run_id,lr,batch_size,weight_decay,grad_clip,final_loss,best_epoch_loss,last_3_avg_loss,runtime_minutes,sample_dir
1,2,0.00010,16,0.0001,1.0,0.027636,0.024348,0.026993,47.612286,gridsearch_samples/run_002_lr0.0001_bs16_wd0.0...
0,1,0.00005,16,0.0001,1.0,0.026906,0.024623,0.027415,47.651605,gridsearch_samples/run_001_lr5e-05_bs16_wd0.00...


Final training

In [79]:
IMG_SIZE = 64
TIMESTEPS = 500
EPOCHS = 3000

param_grid = {
    "lr": [1e-5],
    "batch_size": [16],
    "weight_decay": [1e-4],
    "grad_clip": [1.0],
}

keys = list(param_grid.keys())
configs = [dict(zip(keys, values)) for values in itertools.product(*param_grid.values())]

In [80]:
# -------------------------
# Run training 
# -------------------------

all_results = []

for run_id, config in enumerate(configs, start=1):
    result = train_one_config(config, run_id)
    all_results.append(result)

    with open("final_loss_results.json", "w") as f:
        json.dump(all_results, f, indent=2)

    df = pd.DataFrame(all_results)
    df.to_csv("final_loss_results.csv", index=False)

print("Training complete.")

Saved real samples to gridsearch_samples/run_001_lr1e-05_bs16_wd0.0001_clip1.0/real_epoch_000.png
Real batch stats: min= -1.0 max= 1.0 mean= -0.4527468681335449 std= 0.4115946888923645


Run 1/1:   0%|          | 0/750000 [00:00<?, ?it/s]

Saved generated samples to gridsearch_samples/run_001_lr1e-05_bs16_wd0.0001_clip1.0/generated_run_01_epoch_050_lr_1e-05_bs_16_wd_0.0001_gc_1.0.png

Epoch 50: avg_loss=0.069281
Saved real samples to gridsearch_samples/run_001_lr1e-05_bs16_wd0.0001_clip1.0/real_epoch_050.png
Real batch stats: min= -1.0 max= 1.0 mean= -0.07338367402553558 std= 0.653232753276825
Saved generated samples to gridsearch_samples/run_001_lr1e-05_bs16_wd0.0001_clip1.0/generated_run_01_epoch_100_lr_1e-05_bs_16_wd_0.0001_gc_1.0.png

Epoch 100: avg_loss=0.060976
Saved real samples to gridsearch_samples/run_001_lr1e-05_bs16_wd0.0001_clip1.0/real_epoch_100.png
Real batch stats: min= -1.0 max= 1.0 mean= -0.18381938338279724 std= 0.6180230379104614
Saved generated samples to gridsearch_samples/run_001_lr1e-05_bs16_wd0.0001_clip1.0/generated_run_01_epoch_150_lr_1e-05_bs_16_wd_0.0001_gc_1.0.png

Epoch 150: avg_loss=0.057895
Saved real samples to gridsearch_samples/run_001_lr1e-05_bs16_wd0.0001_clip1.0/real_epoch_150.png
R

In [81]:
# -------------------------
# Resume training
# -------------------------

RESUME_PATH = "gridsearch_samples/run_001_lr0.0001_bs16_wd0.0_clip1.0/best_model.pt"
SAVE_DIR = "resumed_run"
EPOCHS = 2000
SAVE_EVERY = 100
LR = 1e-4

os.makedirs(SAVE_DIR, exist_ok=True)

model = UNet().to(DEVICE)
model.load_state_dict(torch.load(RESUME_PATH, map_location=DEVICE))
print(f"Loaded weights from {RESUME_PATH}")

ema_weights = {k: v.clone() for k, v in model.state_dict().items()}
ema_decay = 0.9999

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.0)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

loader = DataLoader(grid_dataset, batch_size=16, shuffle=True, num_workers=0, pin_memory=False)

pbar = tqdm(total=EPOCHS * len(loader), desc="Resumed training")

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for images in loader:
        if isinstance(images, (list, tuple)):
            images = images[0]
        images = images.to(DEVICE, non_blocking=True)

        t = torch.randint(0, TIMESTEPS, (images.size(0),), device=DEVICE, dtype=torch.long)

        with torch.amp.autocast("cuda", enabled=False):
            noisy_images, noise = add_noise(images, t)
            predicted_noise = model(noisy_images, t)
            loss = F.mse_loss(predicted_noise, noise)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        with torch.no_grad():
            for k, v in model.state_dict().items():
                ema_weights[k].lerp_(v, 1 - ema_decay)

        total_loss += loss.item()
        pbar.set_postfix({"epoch": epoch + 1, "loss": f"{loss.item():.4f}"})
        pbar.update(1)

    scheduler.step()
    avg_loss = total_loss / len(loader)

    if (epoch + 1) % SAVE_EVERY == 0 or (epoch + 1) == EPOCHS:
        print(f"\nEpoch {epoch + 1}: avg_loss={avg_loss:.6f}, lr={scheduler.get_last_lr()[0]:.2e}")

        # Sample with EMA weights
        original = {k: v.clone() for k, v in model.state_dict().items()}
        model.load_state_dict(ema_weights)

        samples = sample(model, n=4, debug=True)
        save_image(samples, os.path.join(SAVE_DIR, f"samples_epoch_{epoch+1:04d}.png"), nrow=2)
        print(f"Saved samples to {SAVE_DIR}/samples_epoch_{epoch+1:04d}.png")

        # Save EMA weights (these are what you want for inference)
        torch.save(ema_weights, os.path.join(SAVE_DIR, "ema_model.pt"))
        # Save training weights too for resuming later
        model.load_state_dict(original)
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, "training_model.pt"))

pbar.close()
print("Done. Final models saved to", SAVE_DIR)

Loaded weights from gridsearch_samples/run_001_lr0.0001_bs16_wd0.0_clip1.0/best_model.pt


Resumed training:   0%|          | 0/500000 [00:00<?, ?it/s]


Epoch 100: avg_loss=0.027669, lr=9.94e-05
t=0400 | x min=-1.500 | x max=1.500 | x mean=-0.251 | x std=0.828 | pred_noise std=0.740
t=0300 | x min=-1.500 | x max=1.500 | x mean=-0.292 | x std=0.808 | pred_noise std=0.730
t=0200 | x min=-1.500 | x max=1.500 | x mean=-0.341 | x std=0.788 | pred_noise std=0.799
t=0100 | x min=-1.500 | x max=1.500 | x mean=-0.386 | x std=0.773 | pred_noise std=0.911
t=0000 | x min=-0.960 | x max=0.948 | x mean=-0.411 | x std=0.756 | pred_noise std=0.909
Before clamp: min= -0.9603496789932251 max= 0.9483373761177063 mean= -0.4109277129173279 std= 0.7561361193656921
Saved samples to resumed_run/samples_epoch_0100.png

Epoch 200: avg_loss=0.026673, lr=9.76e-05
t=0400 | x min=-1.500 | x max=1.500 | x mean=-0.139 | x std=0.835 | pred_noise std=0.743
t=0300 | x min=-1.500 | x max=1.500 | x mean=-0.150 | x std=0.812 | pred_noise std=0.770
t=0200 | x min=-1.500 | x max=1.500 | x mean=-0.168 | x std=0.791 | pred_noise std=0.836
t=0100 | x min=-1.500 | x max=1.500 |